# 🔬 Exploratory Data Analysis — ML Forex Advisor

Notebook ini digunakan untuk eksplorasi data, validasi fitur, dan analisis model sebelum training.

**Urutan penggunaan:**
1. Setup & Load Data
2. Statistik Deskriptif
3. Visualisasi OHLCV
4. Analisis Indikator Teknikal
5. Distribusi Label
6. Feature Importance (setelah training)
7. Analisis Sinyal

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

import yaml

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

SYMBOL    = cfg['data']['primary_symbol']
TIMEFRAME = cfg['data']['timeframes']['primary']

print(f'Config loaded | Symbol: {SYMBOL} | TF: {TIMEFRAME}')

## 1. Load Data

In [ ]:
from data.collector import MT5Collector

collector = MT5Collector(cfg)
connected = collector.connect()

if connected:
    df_raw = collector.get_ohlcv(SYMBOL, TIMEFRAME, 2000, save=False)
else:
    print('MT5 tidak tersedia — menggunakan data simulasi')
    df_raw = collector._generate_dummy_data(2000)

print(f'Data shape  : {df_raw.shape}')
print(f'Date range  : {df_raw.index[0]} → {df_raw.index[-1]}')
df_raw.tail()

## 2. Statistik Deskriptif

In [ ]:
print('=== Statistik OHLCV ===')
print(df_raw.describe().round(5))

print('\n=== Missing values ===')
print(df_raw.isnull().sum())

print('\n=== Return stats ===')
ret = df_raw['close'].pct_change().dropna()
print(f'  Mean return  : {ret.mean()*100:.4f}%')
print(f'  Std return   : {ret.std()*100:.4f}%')
print(f'  Skewness     : {ret.skew():.4f}')
print(f'  Kurtosis     : {ret.kurt():.4f}')
print(f'  Ann. vol     : {ret.std()*np.sqrt(252*24)*100:.2f}%')

## 3. Visualisasi OHLCV

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

n_show = 300
df_plot = df_raw.tail(n_show)

# Price
axes[0].plot(df_plot.index, df_plot['close'], lw=1, color='#378add')
axes[0].fill_between(df_plot.index,
                      df_plot['close'].rolling(20).mean() - df_plot['close'].rolling(20).std(),
                      df_plot['close'].rolling(20).mean() + df_plot['close'].rolling(20).std(),
                      alpha=0.1, color='#378add')
axes[0].set_ylabel('Close Price')
axes[0].set_title(f'{SYMBOL} {TIMEFRAME} — Close Price (last {n_show} bars)')
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(df_plot.index, df_plot['volume'], color='#888780', alpha=0.6, width=0.04)
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

# Returns
ret = df_plot['close'].pct_change()
axes[2].bar(df_plot.index, ret, color=np.where(ret >= 0, '#1d9e75', '#e24b4a'), alpha=0.7, width=0.04)
axes[2].axhline(0, color='gray', lw=0.5)
axes[2].set_ylabel('Return')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Feature Engineering & Analisis Indikator

In [ ]:
from data.feature_engineering import FeatureEngineer

fe = FeatureEngineer(cfg)
df_feat = fe.build_features(df_raw, add_labels=True)
feat_cols = fe.get_feature_columns(df_feat)

print(f'Total fitur  : {len(feat_cols)}')
print(f'Data setelah dropna: {len(df_feat)} baris')
print(f'\nBeberapa nama fitur:')
for c in feat_cols[:20]:
    print(f'  - {c}')

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
df_plot = df_feat.tail(300)

# Price + BB + EMA
axes[0].plot(df_plot.index, df_plot['close'],    color='#378add',  lw=1.5, label='Close')
axes[0].plot(df_plot.index, df_plot['ema_21'],   color='#ba7517',  lw=1,   label='EMA21', alpha=0.8)
axes[0].plot(df_plot.index, df_plot['ema_50'],   color='#7f77dd',  lw=1,   label='EMA50', alpha=0.8)
axes[0].fill_between(df_plot.index, df_plot['bb_upper'], df_plot['bb_lower'],
                     alpha=0.06, color='#378add')
axes[0].legend(fontsize=8)
axes[0].set_ylabel('Price')
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df_plot.index, df_plot['rsi'], color='#7f77dd', lw=1)
axes[1].axhline(70, color='#e24b4a', lw=0.8, ls='--', label='OB 70')
axes[1].axhline(30, color='#1d9e75', lw=0.8, ls='--', label='OS 30')
axes[1].axhline(50, color='gray', lw=0.5, ls=':')
axes[1].set_ylim(0, 100)
axes[1].set_ylabel('RSI')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(df_plot.index, df_plot['macd'],        color='#378add', lw=1, label='MACD')
axes[2].plot(df_plot.index, df_plot['macd_signal'],  color='#e24b4a', lw=1, label='Signal')
axes[2].bar( df_plot.index, df_plot['macd_hist'],
             color=np.where(df_plot['macd_hist'] >= 0, '#1d9e75', '#e24b4a'),
             alpha=0.5, width=0.04)
axes[2].axhline(0, color='gray', lw=0.5)
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

# ATR
axes[3].plot(df_plot.index, df_plot['atr'], color='#ba7517', lw=1)
axes[3].set_ylabel('ATR')
axes[3].grid(True, alpha=0.3)

plt.suptitle(f'Indikator Teknikal — {SYMBOL}', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Distribusi Label

In [ ]:
label_map = {0: 'HOLD', 1: 'BUY', 2: 'SELL'}
label_counts = df_feat['label'].map(label_map).value_counts()
label_pct    = label_counts / len(df_feat) * 100

print('=== Distribusi Label ===')
for lbl, cnt in label_counts.items():
    print(f'  {lbl:<6}: {cnt:>5} ({label_pct[lbl]:.1f}%)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

colors = {'BUY': '#1d9e75', 'SELL': '#e24b4a', 'HOLD': '#888780'}
bar_colors = [colors[l] for l in label_counts.index]
ax1.bar(label_counts.index, label_counts.values, color=bar_colors)
ax1.set_title('Distribusi Label (count)')
ax1.set_ylabel('Jumlah')

ax2.pie(label_counts.values, labels=label_counts.index,
        colors=bar_colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Distribusi Label (%)')

plt.tight_layout()
plt.show()

## 6. Korelasi Fitur dengan Label

In [ ]:
# Top korelasi fitur dengan label
corr = df_feat[feat_cols + ['label']].corr()['label'].drop('label').abs().sort_values(ascending=False)

print('=== Top 20 Fitur (korelasi absolut dengan label) ===')
print(corr.head(20).round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
top20 = corr.head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color='#378add', alpha=0.8)
ax.set_xlabel('Korelasi Absolut dengan Label')
ax.set_title('Top 20 Fitur Berdasarkan Korelasi')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 7. Feature Importance (dari XGBoost yang sudah ditraining)

In [ ]:
from pathlib import Path
xgb_path = Path(cfg['paths']['xgb_model'])

if xgb_path.exists():
    from models.xgboost_model import XGBoostTrainer
    xgb_trainer = XGBoostTrainer(cfg)
    xgb_trainer.load_model()

    top_feats = xgb_trainer.get_top_features(feat_cols, top_n=25)

    names, imps = zip(*top_feats)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(names[::-1], imps[::-1], color='#7f77dd', alpha=0.85)
    ax.set_xlabel('Importance Score')
    ax.set_title('XGBoost Feature Importance (Top 25)')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
else:
    print(f'Model belum ada di {xgb_path}.')
    print('Jalankan: python main.py --mode train terlebih dahulu.')

## 8. Analisis Sinyal vs Return Aktual

In [ ]:
# Berapa rata-rata return aktual per label?
lookahead = cfg['label']['lookahead_bars']
df_analysis = df_feat.copy()
df_analysis['future_return'] = df_analysis['close'].shift(-lookahead) / df_analysis['close'] - 1
df_analysis = df_analysis.dropna(subset=['future_return'])

print(f'=== Return Rata-rata per Label (lookahead={lookahead} bar) ===')
for lbl_id, lbl_name in label_map.items():
    subset = df_analysis[df_analysis['label'] == lbl_id]['future_return']
    if len(subset) > 0:
        print(f'  {lbl_name:<5}: mean={subset.mean()*100:.4f}%  std={subset.std()*100:.4f}%  n={len(subset)}')

fig, ax = plt.subplots(figsize=(10, 5))
for lbl_id, lbl_name in label_map.items():
    subset = df_analysis[df_analysis['label'] == lbl_id]['future_return'] * 100
    if len(subset) > 0:
        ax.hist(subset, bins=40, alpha=0.5, label=lbl_name,
                color=colors.get(lbl_name, 'gray'), density=True)

ax.axvline(0, color='black', lw=1, ls='--')
ax.set_xlabel(f'Return {lookahead} bar ke depan (%)')
ax.set_ylabel('Density')
ax.set_title('Distribusi Return Aktual per Label')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Quick Preprocessing Check
Validasi bahwa tidak ada data leakage antara train dan test.

In [ ]:
from data.preprocessor import Preprocessor

pre = Preprocessor(cfg)
Xt, Xv, Xte, yt, yv, yte = pre.split(df_feat, feat_cols)

print('=== Split Summary ===')
print(f'  Train : {len(Xt):>5} samples | Classes: {dict(zip(*np.unique(yt, return_counts=True)))}')
print(f'  Val   : {len(Xv):>5} samples | Classes: {dict(zip(*np.unique(yv, return_counts=True)))}')
print(f'  Test  : {len(Xte):>5} samples | Classes: {dict(zip(*np.unique(yte, return_counts=True)))}')

print(f'\n=== Scaler Stats (pada train set) ===')
print(f'  Mean range : [{pre.scaler.mean_.min():.4f}, {pre.scaler.mean_.max():.4f}]')
print(f'  Std range  : [{pre.scaler.scale_.min():.4f}, {pre.scaler.scale_.max():.4f}]')

# Cek tidak ada NaN setelah transform
assert not np.any(np.isnan(Xt)), 'NaN di X_train!'
assert not np.any(np.isnan(Xv)), 'NaN di X_val!'
print('\n✅ Tidak ada NaN setelah normalisasi.')

---
**Selesai.** Lanjutkan ke training dengan:
```bash
python main.py --mode train --symbol EURUSD
```